# Batch: Évaluation de granularité dans des noyaux de levures

## 0. Imports requis et paramètres globaux

In [1]:
from napari_yeasts_granularity.operators.data_loader import DataLoader
from napari_yeasts_granularity import FindCellsOperator
from napari_yeasts_granularity import (
    MeasureIntensitiesOperator,
    MeasureShapeOperator,
    MeasureSpotsOperator,
    MeasurementsManager,
    MeasureColocOperator
)
import pandas as pd
import xarray as xr
from pathlib import Path

- `channel_indices`: The rank of each channel in the input images.
    - "brightfield": The transmitted light channel.
    - "main": The channel used to segment nuclei and evaluate the granularity.
    - "secondary": The channel used to evaluate the colocalization with the 'main' channel.
- `images_root`: The root folder containing the hierarchy of images.
- `labels_root`: Where the label maps will be saved.
- `results_root`: Where the CSVs will be saved.
- `base_calibration`: The physical size of voxels for each axis (and 1.0 for time).

In [2]:
channel_indices = {
    'brightfield': 0,
    'main'       : 1,
    'secondary'  : 2
}
images_root = Path("/home/clement/Documents/projects/2292-yeasts-granularity/2026-05-11-tiff")
labels_root = Path("/media/clement/fae534f3-f6ab-41aa-9554-baf3f7b791731/yeasts-granularity")
results_root = Path("/media/clement/fae534f3-f6ab-41aa-9554-baf3f7b791731/results")
base_calibration = {
    'T': 1.0,
    'Z': 0.3,
    'Y': 0.065,
    'X': 0.065
}

## 1. Segmentation des noyaux

### A. Paramètres de segmentation des noyaux

- `kill_borders`: Est-ce qu'on retire les noyaux qui touchent la bordure X, Y et Z ?
- `gaussian_sigma`: Sigma (≈ radius) du flou utilisé lors de la segmentation pour débruiter.
- `use_log`: Est-ce que la fonction logarithme est appliquée à l'image pour essayer d'attraper les noyaux les plus sombres ?
- `log_factor`: Facteur appliqué à l'image avant d'être passée au logarithme. Plus grand == noyaux plus sombres attrapés.
- `min_obj_size`: Taille minimale (en nombre de voxels) qu'un objet doit faire pour ne pas être considéré comme un débris.
- `objects_diam`: En µm, diamètre approximatif d'un noyau. Utile pour le tracking. Si un noyau bouge plus que cette distance, il est considéré perdu.

In [3]:
kill_borders   = True # FindCellsOperator.default_kill_borders()
gaussian_sigma = 1.0  # FindCellsOperator.default_gaussian_sigma()
use_log        = True # FindCellsOperator.default_use_log()
log_factor     = 25.0 # FindCellsOperator.default_log_factor()
min_obj_size   = 100  # FindCellsOperator.default_min_obj_size()
objects_diam   = 1.7  # FindCellsOperator.default_objects_diam()

### B. Segmentation des noyaux

In [4]:
dl = DataLoader(
    group_size=len(channel_indices), 
    group_index=channel_indices['main']
)
dl.set_root_paths(
    images_root, 
    labels_root, 
    results_root
)
dl.set_base_calibration(base_calibration)

while dl.next():
    if dl.labelsExist():
        print(f"Labels already exist for {dl.get_current_image_name()}. Skipping...")
        continue
    image, image_axes = dl.load_image()
    calib = dl.get_calibration(image_axes)

    op = FindCellsOperator()
    op.set_input_image(image, axes=image_axes)
    op.set_calibration(calib)
    op.set_kill_borders(kill_borders)
    op.set_gaussian_sigma(gaussian_sigma)
    op.set_use_log(use_log)
    op.set_log_factor(log_factor)
    op.set_min_obj_size(min_obj_size)
    op.set_objects_diam(objects_diam)
    op.run()

    dl.save_labels(op.get_output_nuclei().values)

Loaded image 2/120: 1 Pus1 + NaCl_002.tif
Labels already exist for 1 Pus1 + NaCl_002.tif. Skipping...
Loaded image 5/120: 1 Pus1 - NaCl_002.tif
Labels already exist for 1 Pus1 - NaCl_002.tif. Skipping...
Loaded image 8/120: 1 Pus1 + NaCl_005.tif
Labels already exist for 1 Pus1 + NaCl_005.tif. Skipping...
Loaded image 11/120: 1 Pus1 - NaCl_005.tif
Labels already exist for 1 Pus1 - NaCl_005.tif. Skipping...
Loaded image 14/120: 1 Pus1 + NaCl_008.tif
Labels already exist for 1 Pus1 + NaCl_008.tif. Skipping...
Loaded image 17/120: 1 Pus1 - NaCl_008.tif
Labels already exist for 1 Pus1 - NaCl_008.tif. Skipping...
Loaded image 20/120: 1 Pus1 + NaCl_011.tif
Labels already exist for 1 Pus1 + NaCl_011.tif. Skipping...
Loaded image 23/120: 1 Pus1 - NaCl_011.tif
Labels already exist for 1 Pus1 - NaCl_011.tif. Skipping...
Loaded image 26/120: 3 Pus1 1 + NaCl 1 puis 4_002.tif
Labels already exist for 3 Pus1 1 + NaCl 1 puis 4_002.tif. Skipping...
Loaded image 29/120: 3 Pus1 1 + NaCl 1 puis 4_005.tif


## 2. Mesures dans les noyaux

### A. Paramètres de mesure

In [5]:
use_intensities_measurements = True
use_shape_measurements       = True
use_spots_count_measurements = True
use_coloc_measurements       = True

spots_min_prominence         = 350.0 # MeasureSpotsOperator.get_default_prominence()

### B. Mesures dans les noyaux

In [6]:
dl1 = DataLoader(
    group_size=len(channel_indices), 
    group_index=channel_indices['main']
)
dl1.set_root_paths(
    images_root, 
    labels_root, 
    results_root
)
dl1.set_base_calibration(base_calibration)

dl2 = DataLoader(
    group_size=len(channel_indices), 
    group_index=channel_indices['secondary']
)
dl2.set_root_paths(
    images_root, 
    labels_root, 
    results_root
)
dl2.set_base_calibration(base_calibration)

while dl1.next() and dl2.next():
    c1, c1_axes = dl1.load_image()
    calib = dl1.get_calibration(c1_axes)
    c2, c2_axes = dl2.load_image()

    c1_arr = xr.DataArray(c1, dims=list(c1_axes))
    c2_arr = xr.DataArray(c2, dims=list(c2_axes))
    labels, label_axes = dl1.load_labels()

    if labels is None or label_axes is None:
        print(f"Skipping measurement.")
        continue

    labels_arr = xr.DataArray(labels, dims=list(label_axes))
    manager = MeasurementsManager()
    manager.set_input_data(labels_arr, c1_arr, calib)

    if use_intensities_measurements:
        mio = MeasureIntensitiesOperator()
        manager.add_operator(mio)
    if use_shape_measurements:
        mso = MeasureShapeOperator()
        manager.add_operator(mso)
    if use_spots_count_measurements:
        mspo = MeasureSpotsOperator()
        manager.add_operator(mspo)
        mspo.set_prominence(spots_min_prominence)
    if use_coloc_measurements:
        mco = MeasureColocOperator()
        manager.add_operator(mco)
        mco.set_secondary_image(c2_arr)

    manager.run()
    measurements = manager.get_merged_measurements()
    measurements_path = dl1.get_results_path("summary")
    measurements.to_csv(measurements_path, index=False)

    controls = manager.get_controls()
    for key, (control, what) in controls.items():
        if what == 'points':
            df = pd.DataFrame(control, columns=["axis-0", "axis-1"])
            spots_path = dl1.get_results_path("spots")
            df.to_csv(spots_path, index=True, index_label="index")
        else:
            print(f"Unknown control type '{what}' for key '{key}'.")

Loaded image 2/120: 1 Pus1 + NaCl_002.tif
Loaded image 3/120: 1 Pus1 + NaCl_003.tif
Running operator: MeasureIntensitiesOperator


100%|██████████| 4/4 [00:30<00:00,  7.75s/it]


Running operator: MeasureShapeOperator


  0%|          | 0/4 [00:00<?, ?it/s]/home/clement/Documents/projects/2292-yeasts-granularity/napari-yeasts-granularity/src/napari_yeasts_granularity/operators/measure_shape_operator.py:27: FutureWarning: `RegionProperties.major_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_major_length` instead. 
  "Sphericity": region.equivalent_diameter / region.major_axis_length if region.major_axis_length > 0 else np.nan
/home/clement/Documents/projects/2292-yeasts-granularity/napari-yeasts-granularity/src/napari_yeasts_granularity/operators/measure_shape_operator.py:27: FutureWarning: `RegionProperties.equivalent_diameter` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.equivalent_diameter_area` instead. 
  "Sphericity": region.equivalent_diameter / region.major_axis_length if region.major_axis_length > 0 else np.nan
/home/clement/miniconda3/envs/napari-cpsam/lib/python3.12/site-p

Running operator: MeasureSpotsOperator


 25%|██▌       | 1/4 [03:00<09:02, 180.79s/it]


KeyboardInterrupt: 